# DRAPE — Feature Store Builder
Runtime → Change runtime type → **T4 GPU**

This notebook downloads the dataset and builds the image embedding index used by the Streamlit app.

In [ ]:
!pip install -q kaggle tqdm pandas scikit-learn

In [ ]:
# Upload kaggle.json from https://www.kaggle.com/settings → API → Create New Token
from google.colab import files
import os

uploaded = files.upload()  # select kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle configured ✓')

In [ ]:
!kaggle datasets download -d paramaggarwal/fashion-product-images-small
!unzip -q fashion-product-images-small.zip -d fashion-dataset
!ls fashion-dataset/
!ls fashion-dataset/images/ | head -5
print('Dataset ready ✓')

In [ ]:
# Upload extract_features.py
files.upload()  # select extract_features.py

In [ ]:
# Build feature store
# max_images=10000 takes ~5 min on T4. Set to 0 for all ~44k images (~20 min)
!python extract_features.py \
    --data_dir ./fashion-dataset \
    --output_dir ./feature_store \
    --batch_size 128 \
    --max_images 10000

In [ ]:
# Verify the output
import numpy as np, json
feats = np.load('feature_store/features.npy')
paths = np.load('feature_store/paths.npy', allow_pickle=True)
with open('feature_store/metadata.json') as f:
    meta = json.load(f)

print(f'Features : {feats.shape}  (should be N × 2048)')
print(f'Paths    : {len(paths)} images')
print(f'Metadata : {len(meta)} records')

In [ ]:
# Download to your machine
import shutil
shutil.make_archive('feature_store', 'zip', 'feature_store')
files.download('feature_store.zip')
print('Downloaded! Unzip to ./feature_store/ in your project folder.')